# 安装OpenClaw

方式1：通过云服务器ECS预置镜像安装，2核2G起步即可支持OpenClaw运行。

方式2：基于容器服务镜像一键部署，零门槛秒级启动并支持弹性扩展。

方式3：依托AI云桌面预置镜像灵活部署，支持Windows、Mac、iOS、Android多端登录。

方式4：自行本地安装：

1）macOS/Linux：
```bash
curl -fsSL https://openclaw.ai/install.sh | bash
```
2）Windows：  
在PowerShell中执行
```bash
iwr -useb https://openclaw.ai/install.ps1 | iex

# 在OpenClaw中配置Coding Plan

步骤1：打开配置文件。  
运行以下命令打开 Web UI，然后在Web UI的左侧菜单栏中选择Config > Raw。
```bash
openclaw dashboard
```

步骤2：修改配置文件。

1）在 JSON 根对象中加入如下 models 配置（如果已存在则替换）。

- 请将 <YOUR_API_KEY> 替换为您的 Coding Plan 专属API Key

- 在baseUrl中填写Coding Plan的BaseUrl。目前套餐仅支持贵阳基地二区，使用openai接口协议时，填写https://aigw-gzgy2.cucloud.cn:8443/v1

```json
"models": {
  "mode": "merge",
  "providers": {
    "unicom-cloud": {
      "baseUrl": "https://aigw-gzgy2.cucloud.cn:8443/v1",
      "apiKey": "<YOUR_API_KEY>",
      "api": "openai-completions",
      "models": [
        {
          "id": "MiniMax-M2.5",
          "name": "MiniMax-M2.5",
          "reasoning": false,
          "input": ["text"],
          "cost": { "input": 0, "output": 0, "cacheRead": 0, "cacheWrite": 0 },
          "contextWindow": 202752,
          "maxTokens": 16384
        }
      ]
    }
  }
}
```

2）找到 agents.defaults 对象，并替换或添加以下两个字段：、
```
"model": {
  "primary": "unicom-cloud/MiniMax-M2.5"
},
"models": {
  "unicom-cloud/MiniMax-M2.5": {}
}
```

步骤3：保存配置  
如果在Web UI中修改，先单击右上角 Save 保存，然后单击 Update来使配置生效。  
如果在终端中修改，先保存文件并退出，然后运行以下命令来使配置生效。  
```bash
openclaw gateway restart
```

步骤4：使用  
在终端运行以下命令，可以用 Web UI 的方式使用 OpenClaw。
```bash
openclaw dashboard
```

三、常用命令

| 命令                        | 作用           | 说明                                                                 |
|-----------------------------|----------------|----------------------------------------------------------------------|
| `openclaw onboard`          | 启动配置向导   | 交互式配置，如果遇到问题可以重新执行引导                             |
| `openclaw gateway install`  | 安装服务       | 同时启动服务，并设置开机自启                                         |
| `openclaw gateway start`    | 启动服务       | 需要先执行 `install` 命令                                            |
| `openclaw gateway stop`     | 停止服务       | 会卸载服务（注意与 `start` 的依赖关系）                             |
| `openclaw gateway status`   | 查看状态       | 检查服务是否正在运行                                                 |
| `openclaw logs --follow`    | 查看实时日志   | 跟踪执行过程中的日志输出（类似 `tail -f`）                          |


# 方式一
以联通云为例

1. 购买云服务器ECS，并选择OpenClaw镜像。  

    规格大于等于2C2G即可。  

    该云服务器创建成功后，已具备OpenClaw应用以及初始环境。

2. 获取API Key

    在联通云AI服务平台选择“服务接入”，点击“创建服务”。选择您所需要的大模型类型，获得API Key。  
    或购买CodingPlan，获得API Key。

3. OpenClaw配置（采用联通云内置脚本一键部署）（自定义配置见 https://support.cucloud.cn/document/127/571/128.html?id=128&arcid=6999 ）

    通过SSH或VNC连接进入云服务器ECS
    ```bash
    ssh 用户名@公网IP地址
    # 然后输入你的密码
    ```

    注意：为保证安全使用OpenClaw，OpenClaw云服务器ECS在创建时，系统为您分配默认专有网络VPC、默认子网、默认安全组，  
    所以在创建完成时，请您进入云服务器后，执行以下命令，检查并配置DNS解析服务：
    ```bash
    vim /etc/resolv.conf
    ```
    在打开的文件中，确认是否存在nameserver配置项。若未配置或需要修改，请添加以下内容（以Google公共DNS为例）:  nameserver 8.8.8.8  

    先按i进入编辑模式，在文件中添加 nameserver 8.8.8.8 ，再按esc退出编辑模式，输入:wq（或:x）保存并退出（注意：输入法要是英文）

    下载最新的脚本并赋予执行权限
    ```bash
    curl -o update-openclaw-config.py https://console.cucloud.cn/console/subApp/woc-ecs-beta/assets/update-openclaw-config.py
    chmod +x update-openclaw-config.py
    ```

    运行脚本，按提示进行配置（配置钉钉见 https://support.cucloud.cn/document/127/571/128.html?id=128&arcid=7008 ）
    ```bash
    ./update-openclaw-config.py
    ```

    之后可以在终端输入以下内容启动OpenClaw并交互
    ```bash
    python3 -m openclaw.cli
    ```





# 常用命令
| 命令                          | 作用          | 说明                                                                 |
|-------------------------------|---------------|----------------------------------------------------------------------|
| `vim /etc/resolv.conf`        | 配置 DNS      | 创建 OpenClaw 的 ECS 实例后，配置 DNS 解析服务（如修改 nameserver）。 |
| `ps -ef \| grep open`         | 查询服务状态  | 检查 OpenClaw 相关进程是否已启动（如 `openclaw-gateway`）。          |
| `openclaw gateway run`        | 启动服务      | 正常启动 OpenClaw 服务（前台运行，终端会持续输出日志）。             |
| `openclaw gateway --force`    | 强制重启服务  | 杀死旧进程并重新启动 OpenClaw 服务（适用于服务异常时强制恢复）。      |
| `openclaw logs --follow`      | 实时查看日志  | 跟踪 OpenClaw 执行过程中的日志输出（类似 `tail -f`）。               |

![image.png](https://support.cucloud.cn/upload/cms/content/editor/1770886905520.png)

| **功能分类**       | **命令**                          | **说明**                                                                 |
|--------------------|-----------------------------------|--------------------------------------------------------------------------|
| **基础操作**       | `openclaw start`                  | 启动 OpenClaw 服务                                                       |
|                    | `openclaw stop`                   | 停止 OpenClaw 服务                                                       |
|                    | `openclaw restart`                | 重启 OpenClaw 服务                                                       |
|                    | `openclaw status`                 | 查看服务运行状态                                                         |
| **网关管理**       | `openclaw gateway start`          | 启动网关服务                                                             |
|                    | `openclaw gateway stop`           | 停止网关服务                                                             |
|                    | `openclaw gateway restart`        | 重启网关服务                                                             |
|                    | `openclaw gateway status`         | 查看网关状态（包括监听端口、服务健康度等）                               |
| **配置管理**       | `openclaw config get <path>`      | 查看指定配置项的值（如 `config get models.default`）                     |
|                    | `openclaw config set <path> <value>` | 修改配置项（如 `config set gateway.port 18789`）                         |
|                    | `openclaw configure`              | 交互式配置向导（模型、通道、技能等）                                     |
| **模型与技能管理** | `openclaw models list`            | 列出所有可用模型                                                         |
|                    | `openclaw models set <model>`     | 切换默认模型（如 `models set claude-3-5`）                               |
|                    | `openclaw skills list`            | 列出所有可用技能                                                         |
|                    | `openclaw skills enable <name>`   | 启用指定技能（如 `skills enable file-organizer`）                        |
| **日志与诊断**     | `openclaw logs`                   | 查看实时日志                                                             |
|                    | `openclaw logs --tail=100`        | 查看最近 100 行日志                                                      |
|                    | `openclaw doctor`                 | 系统诊断与自动修复（推荐在遇到问题时运行）                               |
| **通道管理**       | `openclaw channels list`          | 列出已连接的聊天通道（如 Telegram、Discord 等）                          |
|                    | `openclaw channels add --channel telegram` | 添加 Telegram 通道（需提前配置 API Token）                          |
| **内存管理**       | `openclaw memory search "关键词"`  | 搜索长期记忆或每日日志中的内容                                           |
| **定时任务**       | `openclaw cron add --name "任务名" --cron "0 9 * * *" --message "提醒内容"` | 添加每日定时任务（如早上 9 点的提醒） |
| **高级命令**       | `openclaw dashboard`              | 打开 Web 控制面板（默认地址 `http://127.0.0.1:18789`）                   |
|                    | `openclaw update`                 | 检查并更新 OpenClaw 到最新版本                                           |
|                    | `openclaw reset`                  | 重置本地配置（谨慎使用，会清除所有自定义设置）                           |


# 常见问题

1. 若您在使用OpenClaw时，出现“no output”字样：

    可能原因为客户端无法访问大模型，建议检查虚拟机与大模型的连通性，如未配置DNS导致与大模型无法链接。

2. 如果下载最新的脚本赋予执行权限时报如下错误，则可通过配置DNS解析进行解决。
![image.png](https://support.cucloud.cn/upload/cms/content/editor/1772521526648.png)

# 其他
## OpenClaw 的 Web 控制面板
```bash
root@open:~# openclaw dashboard

🦞 OpenClaw 2026.2.9 (33c75cb) — Turning "I'll reply later" into "my bot replied instantly".

Dashboard URL: http://127.0.0.1:18789/#token=b0cafaae9480bbfce097f058f7c3a52c7e5d5250ea9cefc4
```
在 你的本地电脑（而非服务器）的终端中运行以下命令，将服务器的 18789 端口映射到本地的 18789 端口：
```bash
1ssh -N -L 18789:127.0.0.1:18789 root@192.168.1.238
```
- -N：不执行远程命令（仅端口转发）。
- -L 18789:127.0.0.1:18789：将本地 18789 端口转发到服务器的 127.0.0.1:18789。
- root@192.168.1.238：服务器的 SSH 地址（替换为你的实际 IP）。

保持 SSH 隧道运行，然后在本地浏览器中访问以下 URL：http://localhost:18789/